# GPU performance: compare the baseline and optimized code

Select an available GPU under **Runtime > Change runtime type** and run the
cells in order. This notebook uses the TensorFlow installation already in
Colab. It does not download ForTy or require its class mapping.

Both revisions use identical synthetic inputs and one shared checkpoint.
Inference uses float32 and training uses mixed float16. Each workload runs
in a fresh process, with five warmups, 30 measured steps and three rounds in
alternating revision order. Forward agreement is checked before timing.

This measures synthetic model compute and TensorFlow allocator memory. It
excludes imports, tracing, data IO, preprocessing, checkpoint loading and
exports. It does not measure accuracy or total GPU VRAM. GPU availability
and runtime duration are not guaranteed. Keep the downloaded diagnostic ZIP.


In [ ]:
import datetime
import json
import os
from pathlib import Path
import subprocess
import sys
import uuid
import zipfile

BASELINE_COMMIT = "82723f3183231a36e736c1c118a866aed5ad9d6b"
CANDIDATE_COMMIT = "f49b0042e35593fb120bf97bb292d0de8416fd4d"
REPOSITORY = "https://github.com/CCC917111/geospatial-mamba-deforestation.git"
ROUNDS = 3
STEPS = 30
WARMUP = 5
TILE_SIZE = 128
BATCH_SIZE = 1
# Optional: an existing full-default-model checkpoint in this runtime.
# None creates one seeded synthetic-initialization checkpoint for both revisions.
WEIGHTS = None

WORKSPACE = Path("/content/geospatial-performance")
WORKSPACE.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
identifier = stamp + "-" + uuid.uuid4().hex[:8]
SESSION_ROOT = WORKSPACE / ("reports-" + identifier)
SESSION_ROOT.mkdir(exist_ok=False)
SOURCE_ROOT = WORKSPACE / ("sources-" + identifier)
SOURCE_ROOT.mkdir(exist_ok=False)
RUN_ROOT = SESSION_ROOT / "comparison"
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / "cache/matplotlib"))
os.environ.setdefault("KERAS_HOME", str(WORKSPACE / "cache/keras"))
os.environ.setdefault("XDG_CACHE_HOME", str(WORKSPACE / "cache"))
print("Reports:", SESSION_ROOT)

packages = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(SESSION_ROOT / "packages.txt").write_text(packages)


In [ ]:
SOURCES = {}
for label, commit in (("baseline", BASELINE_COMMIT), ("candidate", CANDIDATE_COMMIT)):
    destination = SOURCE_ROOT / label
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY, str(destination)], check=True)
    subprocess.run(["git", "checkout", "--detach", commit], cwd=destination, check=True)
    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=destination, text=True).strip()
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=destination, text=True)
    if head != commit or dirty:
        raise RuntimeError("Source revision mismatch or modified checkout")
    SOURCES[label] = destination
(SESSION_ROOT / "source-revisions.json").write_text(json.dumps({
    "baseline": BASELINE_COMMIT, "candidate": CANDIDATE_COMMIT,
    "rounds": ROUNDS, "steps": STEPS, "warmup": WARMUP,
    "tile_size": TILE_SIZE, "batch_size": BATCH_SIZE,
}, indent=2))


In [ ]:
def archive_reports():
    destination = SESSION_ROOT.parent / (SESSION_ROOT.name + ".zip")
    saved, omitted, total = [], [], 0
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(SESSION_ROOT.rglob("*")):
            if path.is_symlink() or not path.is_file():
                continue
            if path.suffix.lower() not in {".json", ".txt", ".log", ".csv"}:
                continue
            size = path.stat().st_size
            name = str(path.relative_to(SESSION_ROOT))
            if size > 4 * 1024 * 1024 or total + size > 16 * 1024 * 1024:
                omitted.append({"path": name, "bytes": size})
                continue
            archive.write(path, name)
            saved.append(name)
            total += size
        archive.writestr("archive-index.json", json.dumps({
            "saved": saved, "omitted_for_size": omitted,
            "scope": "Diagnostic metadata only; checkpoint and array files remain in the runtime.",
        }, indent=2))
    if destination.stat().st_size >= 32 * 1024 * 1024:
        raise RuntimeError("Diagnostic ZIP exceeds the transfer limit")
    print("Diagnostic ZIP:", destination)
    return destination


In [ ]:
command = [sys.executable, str(SOURCES["candidate"] / "scripts/compare_performance.py"),
           "--baseline-root", str(SOURCES["baseline"]),
           "--candidate-root", str(SOURCES["candidate"]),
           "--run-root", str(RUN_ROOT), "--device", "/GPU:0",
           "--training-precision", "mixed_float16", "--rounds", str(ROUNDS),
           "--steps", str(STEPS), "--warmup", str(WARMUP),
           "--tile-size", str(TILE_SIZE), "--batch-size", str(BATCH_SIZE)]
if WEIGHTS is not None:
    command += ["--weights", str(Path(WEIGHTS).expanduser().resolve())]
(SESSION_ROOT / "command.json").write_text(json.dumps(command, indent=2))
try:
    with (SESSION_ROOT / "driver.log").open("x") as log:
        completed = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    if completed.returncode:
        raise RuntimeError("Performance run failed; retain the ZIP and driver.log for diagnosis")
    report = json.loads((RUN_ROOT / "comparison.json").read_text())
    print(json.dumps(report["summary"], indent=2))
finally:
    ARCHIVE = archive_reports()


The archive contains raw timing samples, forward agreement checks, source
identities, fixture/checkpoint hashes, training update counts and allocator
memory measurements. Weight files and input arrays are intentionally excluded
so the diagnostic upload stays small. They remain in the runtime if needed.

If a run fails, retain the archive and logs. Start a fresh session directory
for a retry. Runtime reclamation can prevent this final download from running.


In [ ]:
from google.colab import files
ARCHIVE = archive_reports()
files.download(str(ARCHIVE))
